# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the "Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells" dataset using the `mlcroissant` library, following FAIR data principles.

### Dataset Source
The dataset Croissant schema is accessible at:

**https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json**

In [ ]:
# Ensure `mlcroissant` is installed
!pip install -U mlcroissant

## 1. Data Loading
Load the metadata and access the records from the Croissant dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")
print(f"License: {metadata.license}")

## 2. Data Overview
Let's enumerate all available **record sets** and their fields. For proper use of the Croissant specification, all references will be made using their `@id`.

In [ ]:
# List all record sets by @id and display their fields/columns (if exist)
record_sets = []
if hasattr(metadata, "record_sets"):
    for rs in metadata.record_sets:
        print(f"Record set name: {getattr(rs, 'name', 'N/A')}")
        print(f"@id: {rs.id}")
        print("Fields/columns (with @id):")
        if hasattr(rs, "fields"):
            for field in rs.fields:
                print(f" - {getattr(field, 'name', 'N/A')}: {field.id}")
        if hasattr(rs, "columns"):
            for col in rs.columns:
                print(f" - {getattr(col, 'name', 'N/A')}: {col.id}")
        record_sets.append(rs.id)
        print()
else:
    print("No record sets found in metadata.")

## 3. Data Extraction
Load each available record set as a DataFrame for analysis.
Below, we use the record set `@id`s determined above.

In [ ]:
# If no record_sets found in metadata, try introspecting from dataset.records API
if not record_sets:
    # Use Croissant Python API to list all available record sets
    record_sets = dataset.record_sets()

print("Available record set @id's:")
print(record_sets)

dataframes = {}

for rs_id in record_sets:
    print(f"Loading records for: {rs_id}")
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f" - Columns for {rs_id}: {df.columns.tolist()}")
            print(df.head(2))
        else:
            print(f" - No records found for {rs_id}.")
    except Exception as e:
        print(f" - Failed to load {rs_id}: {e}")

# Pick the first record set for demonstration
if record_sets:
    first_record_set = record_sets[0]
    print(f"\nWorking with record set: {first_record_set}")
    print(dataframes[first_record_set].head())
else:
    first_record_set = None

## 4. Exploratory Data Analysis (EDA)
Let's examine numeric fields (`float` or `integer` types) and demonstrate filtering, normalization, and grouping.

In [ ]:
import numpy as np
if first_record_set is not None:
    df = dataframes[first_record_set]
    # Find numeric columns
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if not numeric_fields:
        # Try coercing all columns: Croissant may have loaded as all strings
        for col in df.columns:
            try:
                df[col] = pd.to_numeric(df[col])
            except Exception:
                pass
        numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    print("Numeric columns detected:", numeric_fields)

    # Choose the first numeric field if available
    if numeric_fields:
        numeric_field = numeric_fields[0]
        print(f"Using numeric field: {numeric_field}")
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        # Filter records
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records with {numeric_field} > {threshold:0.2f}:")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Group by most common string column
        group_field = None
        for col in df.columns:
            if col != numeric_field and df[col].dtype == object:
                group_field = col
                break
        if group_field:
            print(f"\nGrouping by '{group_field}':")
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(grouped_df.head())
    else:
        print("No numeric fields detected to analyze.")
else:
    print("No data available in record sets for EDA.")

## 5. Visualization
Visualize a numeric field's distribution and group mean by a categorical field.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if first_record_set is not None and numeric_fields:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

    if group_field:
        group_means = df.groupby(group_field)[numeric_field].mean().reset_index()
        plt.figure(figsize=(10,4))
        sns.barplot(y=group_field, x=numeric_field, data=group_means, orient='h')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to:
- Load Croissant dataset metadata and records using their logical `@id` references,
- Introspect record sets and their schema,
- Load records into pandas DataFrames for further exploration,
- Perform basic EDA (filtering, normalization, grouping) using numeric fields,
- Visualize data distributions and group-level statistics.

This workflow highlights the power and versatility of the Croissant format for reproducible and FAIR biomedical data analytics.